setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
BASE_DIR = os.path.dirname(os.getcwd())

DATA_DIR = os.path.join(
    BASE_DIR,
    "data"
)

RAW_DATA_DIR = os.path.join(
    DATA_DIR,
    "raw"
)

PROCESSED_DATA_DIR = os.path.join(
    DATA_DIR,
    "processed"
)

os.makedirs(
    PROCESSED_DATA_DIR,
    exist_ok=True
)

print("Base Directory:", BASE_DIR)
print("Raw Data Directory:", RAW_DATA_DIR)
print("Processed Data Directory:", PROCESSED_DATA_DIR)

In [ ]:
raw_files = os.listdir(RAW_DATA_DIR)

print("Available raw files:")

for file in raw_files:
    print("-", file)

In [ ]:
DATA_FILE = os.path.join(
    RAW_DATA_DIR,
    "ecommerce_data.csv"
)

df = pd.read_csv(DATA_FILE)

In [ ]:
print("Dataset loaded successfully.")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.sample(
    min(5, len(df)),
    random_state=42
)

In [ ]:
rows, columns = df.shape

print(f"Number of Rows: {rows}")
print(f"Number of Columns: {columns}")

In [ ]:
print("Dataset Columns:\n")

for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
missing_values = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().sum()
        / len(df)
        * 100
    ).round(2)
})

missing_values

In [ ]:
missing_values[
    missing_values["Missing Values"] > 0
]

In [ ]:
duplicate_count = df.duplicated().sum()

print(
    "Total Duplicate Rows:",
    duplicate_count
)

In [ ]:
numerical_summary = df.describe()

numerical_summary

In [ ]:
df.describe(
    include="all"
)

In [ ]:
numerical_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = df.select_dtypes(
    include="object"
).columns.tolist()

print("Numerical Columns:")
print(numerical_columns)

print("\nCategorical Columns:")
print(categorical_columns)

In [ ]:
unique_values = pd.DataFrame({
    "Column": df.columns,
    "Unique Values": [
        df[column].nunique()
        for column in df.columns
    ]
})

unique_values

In [ ]:
for column in categorical_columns:

    print("\n" + "=" * 60)

    print(f"Column: {column}")

    print("=" * 60)

    print(
        df[column]
        .value_counts(
            dropna=False
        )
        .head(20)
    )

In [ ]:
possible_date_columns = [
    column
    for column in df.columns
    if "date" in column.lower()
    or "time" in column.lower()
]

print(
    "Possible Date Columns:",
    possible_date_columns
)

In [ ]:
for column in possible_date_columns:

    df[column] = pd.to_datetime(
        df[column],
        errors="coerce"
    )

print("Date conversion completed.")

In [ ]:
df.dtypes

In [ ]:
for column in possible_date_columns:

    print(f"\nDate Column: {column}")

    print(
        "Earliest Date:",
        df[column].min()
    )

    print(
        "Latest Date:",
        df[column].max()
    )

In [ ]:
date_missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing Values": df.isnull().sum()
})

date_missing_report[
    date_missing_report["Missing Values"] > 0
]

In [ ]:
memory_usage = (
    df.memory_usage(
        deep=True
    ).sum()
    / 1024 ** 2
)

print(
    f"Dataset Memory Usage: {memory_usage:.2f} MB"
)

In [ ]:
column_profile = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-Null Count": df.notnull().sum().values,
    "Missing Count": df.isnull().sum().values,
    "Unique Values": df.nunique().values
})

column_profile

In [ ]:
numeric_profile = pd.DataFrame({
    "Minimum": df[numerical_columns].min(),
    "Maximum": df[numerical_columns].max(),
    "Mean": df[numerical_columns].mean(),
    "Median": df[numerical_columns].median()
})

numeric_profile

In [ ]:
outlier_results = []

for column in numerical_columns:

    Q1 = df[column].quantile(0.25)

    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR

    upper_bound = Q3 + 1.5 * IQR

    outlier_count = (
        (
            df[column] < lower_bound
        )
        |
        (
            df[column] > upper_bound
        )
    ).sum()

    outlier_results.append(
        {
            "Column": column,
            "Outlier Count": outlier_count,
            "Outlier Percentage": round(
                outlier_count
                / len(df)
                * 100,
                2
            )
        }
    )

outlier_report = pd.DataFrame(
    outlier_results
)

outlier_report.sort_values(
    "Outlier Count",
    ascending=False
)

In [ ]:
likely_id_columns = []

for column in df.columns:

    unique_ratio = (
        df[column].nunique()
        / len(df)
    )

    if (
        unique_ratio > 0.90
        or "id" in column.lower()
    ):

        likely_id_columns.append(
            column
        )

print("Likely ID Columns:")
print(likely_id_columns)

In [ ]:
dataset_overview = {
    "Total Rows": len(df),
    "Total Columns": len(df.columns),
    "Duplicate Rows": df.duplicated().sum(),
    "Total Missing Values": df.isnull().sum().sum(),
    "Numerical Columns": len(numerical_columns),
    "Categorical Columns": len(categorical_columns),
    "Likely ID Columns": len(likely_id_columns)
}

for key, value in dataset_overview.items():

    print(
        f"{key}: {value}"
    )

In [ ]:
data_quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Missing Percentage": (
        df.isnull().sum()
        / len(df)
        * 100
    ).round(2).values,
    "Unique Values": df.nunique().values
})

data_quality_report

In [ ]:
data_quality_report = (
    data_quality_report
    .sort_values(
        "Missing Values",
        ascending=False
    )
)

data_quality_report

In [ ]:
report_path = os.path.join(
    PROCESSED_DATA_DIR,
    "data_quality_report.csv"
)

data_quality_report.to_csv(
    report_path,
    index=False
)

print(
    "Data quality report saved successfully."
)

print(
    "Saved at:",
    report_path
)

In [ ]:
column_profile_path = os.path.join(
    PROCESSED_DATA_DIR,
    "column_profile.csv"
)

column_profile.to_csv(
    column_profile_path,
    index=False
)

print(
    "Column profile saved successfully."
)

In [ ]:
numeric_profile_path = os.path.join(
    PROCESSED_DATA_DIR,
    "numerical_profile.csv"
)

numeric_profile.to_csv(
    numeric_profile_path
)

print(
    "Numerical profile saved successfully."
)

In [ ]:
print("=" * 60)

print(
    "DATA UNDERSTANDING COMPLETED"
)

print("=" * 60)

print(
    f"Dataset Shape: {df.shape}"
)

print(
    f"Total Missing Values: {df.isnull().sum().sum()}"
)

print(
    f"Duplicate Rows: {df.duplicated().sum()}"
)

print(
    f"Numerical Columns: {len(numerical_columns)}"
)

print(
    f"Categorical Columns: {len(categorical_columns)}"
)

print(
    f"Likely ID Columns: {len(likely_id_columns)}"
)

print("=" * 60)

print(
    "Generated Files:"
)

print(
    "- data_quality_report.csv"
)

print(
    "- column_profile.csv"
)

print(
    "- numerical_profile.csv"
)

print("=" * 60)